# Promtior - Sistema de reservas de salas

Este proyeto es un sistema reservas de salas de reunió. Un usuario puede ver salas, consultar disponibilidad y crear, reprogramar, cancelar o unir sus reservas **conversando con un chatbot**. Un administrador gestiona las salas, ve la agenda semanal y configura las reglas de duración.

### Índice

1. Visión general: por qué una arquitectura en capas
2. Capa de presentación (Controller): por qué Thymeleaf dentro del mismo servidor
3. Capa de servicio: por qué la lógica de negocio vive en los `Service`
4. Chatbot / IA: por qué OpenAI + Spring AI y cómo funciona
5. Capa de persistencia (Repository): PostgreSQL, Flyway, JPA, `Auditable`

## 1. Visión general: Una arquitectura en capas

El sistema se encuentra separa en capas con una separacion clara de responsabilidades

| Capa | Responsabilidad |
|---|---|
| Controller / Tool | Recibir el pedido, validar su forma, identificar al usuario |
| Service | Reglas de negocio|
| Repository | Leer y escribir en la base | 

**Organización del código.** Los paquetes son horizontales, por tipo de archivo (`controller/`, `service/`, `repository/`, `entity/`, `dto/`, `mapper/`…). Para un proyecto de este tamaño hace que sea fácil saber dónde buscar algo y que cada capa se vea completa de un vistazo.



## 2. Capa de presentación (Controller): por qué Thymeleaf dentro del mismo servidor

### El problema
Hacía falta una interfaz para tres cosas: iniciar sesión, gestionar reservas (incluido el chat) y administrar salas y reglas. La pregunta era si construir un **frontend separado** (React, Angular, etc., consumiendo la API) o **renderizar las pantallas desde el propio backend**.

### La decisión: Thymeleaf en el mismo servidor
Las pantallas (`login`, `home`, `chat`, `my-bookings`, `admin-rooms`, `admin-schedule`, `admin-settings`) son plantillas HTML que Spring completa en el servidor con datos del `Model`.

**Por qué, pensando en este proyecto:**

- **Un solo artefacto para desplegar.** Todo sale en un único `.jar` y una única imagen Docker a Railway. Un front separado implica otro proyecto, otro build, otro deploy, otra URL y configurar CORS entre ambos.
- **La UI es mayormente formularios y listados.** Listar salas, filtrar la agenda de la semana, editar la capacidad de una sala: no necesita un framework de SPA. El único punto realmente dinámico es el chat, que se resolvió con un `fetch` a `/chat/messages` desde la misma página.
- **La seguridad web sale "gratis".** Con Thymeleaf se usa el login por formulario, sesión y protección CSRF de Spring Security tal cual vienen. Con un front separado habría que manejar el token en el navegador (dónde guardarlo, cómo refrescarlo).
- **Los controllers llaman a los services directamente**, sin pasar por HTTP. No hay una capa intermedia de cliente API que mantener.
- **Validación con mensajes en la misma pantalla.** En las pantallas de admin, `@Valid` + `BindingResult` permiten volver a mostrar el formulario con el error al lado del campo, sin escribir JavaScript:

```java
@PostMapping
public String createRoom(@Valid @ModelAttribute("newRoom") RoomRequest request,
                         BindingResult bindingResult, ...) {
   if (bindingResult.hasErrors()) { ... return "admin-rooms"; }   // vuelve a la vista con los errores
   ...
}
```

### No se cierra la puerta a un front separado
La **API REST con JWT sigue existiendo** en paralelo (`BookingController`, `RoomController`, …). Si mañana se quiere una app móvil o un front independiente, puede consumirla sin tocar la lógica, porque esa lógica está en los services, no en los controllers.

### Controllers "delgados"
Tanto los `@Controller` de Thymeleaf como los `@RestController` solo traducen la entrada y delegan:

```java
@PostMapping
public BookingResponse create(@Valid @RequestBody BookingRequest request,
                              @AuthenticationPrincipal User currentUser) {
   return bookingService.create(request, currentUser.getId());
}
```

`@AuthenticationPrincipal` entrega el usuario logueado (venga de la sesión o del JWT), así el controller nunca confía en un `userId` que mande el cliente.

### Costos aceptados
- La UI y el backend quedan acoplados en el mismo deploy: un cambio visual implica redesplegar el backend.
- Cada navegación recarga la página; para una herramienta interna de reservas es aceptable.

## 3. Capa de servicio: por qué la lógica de negocio vive en los `Service`

### El problema
Reservar una sala parece simple, pero tiene muchas reglas:

| Regla | Dónde se valida |
|---|---|
| La sala existe y está **habilitada** | `validateRoomActive` |
| La cantidad de asistentes no supera la **capacidad** de la sala | `validateAttendeeCount` |
| No se **solapa** con otra reserva activa de la misma sala | `validateNoOverlap` (consulta JPQL) |
| El inicio cae en la **grilla horaria** (ej. cada 30 min desde las 00:00) | `validateSlotAlignment` |
| La duración es múltiplo de la grilla y no supera el **máximo** | `validateSlotCount` |
| La fecha es **futura** | `validateCurrentDate` |
| Solo el **dueño** puede modificar, cancelar o unir su reserva | `validateOwner` |

Estas reglas las necesitan **la API, la web y el chatbot**. Además, la grilla y la duración máxima **no son fijas**: las configura un admin desde `/admin/settings` y se guardan en la tabla `booking_settings`.

### La decisión: encapsular todas las reglas en `BookingService`, `RoomService`, `BookingSettingsService`

```java
@Transactional(rollbackFor = Exception.class)
public BookingResponse create(@Valid BookingRequest request, UUID userId) {
   validateSlotCount(request.startTime(), request.endTime());
   Room room = getRoomOrThrow(request.roomId());
   User user = getUserOrThrow(userId);
   validateRoomActive(room);
   validateAttendeeCount(request.attendeeCount(), room);
   validateNoOverlap(room.getId(), request.startTime(), request.endTime(), null);
   validateCurrentDate(request.startTime());
   ...
   return bookingMapper.toResponse(bookingRepository.save(booking));
}
```

**En qué ayuda, concretamente:**

- **Una sola fuente de verdad.** El chatbot no tiene reglas propias: `ChatToolsService.createBooking` arma un `BookingRequest` y llama a `bookingService.create(...)`, igual que el `BookingController`. Si el LLM intenta algo inválido, recibe la misma `BusinessException` que recibiría un usuario de la API.
- **Seguridad que no depende del cliente.** "Solo reservas propias" se valida en el service con el `userId` del usuario autenticado. Aunque el chatbot reciba un `bookingId` ajeno, el backend lo rechaza.
- **Reglas configurables.** Como las reglas están en código Java y no en la base, pueden leer la configuración vigente (`bookingSettingsService.getSettings()`) y devolver mensajes claros en español ("El horario de inicio debe caer en la grilla de slots de 30 minutos"). Un `CHECK` en la base no podría leer esa configuración ni dar un mensaje útil (ver sección 5).
- **Atomicidad.** Operaciones como `unify` (extender una reserva y cancelar la otra) llevan `@Transactional`: o se aplican los dos cambios o ninguno. Las lecturas no llevan transacción; solo los métodos que escriben.
- **Errores con significado.** Los services lanzan `ResourceNotFoundException` (→ 404) o `BusinessException` (→ 422), y un único `GlobalExceptionHandler` los convierte en respuestas estándar (sección 6).


## 4. Chatbot / IA: por qué OpenAI + Spring AI y cómo funciona

Se quiere poder gestionar reservas **en lenguaje natural**: *"reservame la sala A mañana de 10 a 11 para la daily, somos 4"*. Eso implica:

1. Entender la intención y extraer datos (sala, fecha relativa, horario, asistentes, motivo).
2. **Ejecutar operaciones reales** en el sistema (consultar disponibilidad, crear, cancelar…).
3. Recordar el contexto de la conversación ("¿y a las 11?").
4. Hacerlo **sin que el modelo invente datos ni saltee reglas**.

### ¿Por qué un LLM de OpenAI?
- Necesitábamos un modelo con **tool calling** (llamado de funciones) confiable: que decida cuándo llamar a los servicios y con qué argumentos. Los modelos de OpenAI lo soportan de forma madura.
- Se consume como **API administrada**: no hay que hospedar ni mantener un modelo propio, solo configurar `OPENAI_API_KEY`.
- El modelo se elige por configuración (`spring.ai.openai.chat.model`), sin tocar código.

### ¿Por qué Spring AI y no llamar a la API de OpenAI "a mano"?
La alternativa "nativa" era usar el SDK de OpenAI o hacer requests HTTP directos. En ese caso habría que programar a mano:

| Qué haría falta implementar | Qué da Spring AI |
|---|---|
| Describir cada función en JSON Schema para el modelo | Se genera solo a partir de `@Tool` y `@ToolParam` sobre métodos Java |
| Detectar que el modelo pidió una función, parsear los argumentos, invocarla y devolverle el resultado, en un loop | `ChatClient` hace ese ciclo automáticamente |
| Guardar y reenviar el historial de cada conversación | `ChatMemory` + `MessageChatMemoryAdvisor` |
| Convertir `LocalDateTime`, `UUID`, records, etc. a/desde JSON | Automático |
| Integrarlo con los beans de Spring, la configuración y la sesión del usuario | Es un starter más de Spring Boot |

Además, Spring AI **abstrae el proveedor**: cambiar a otro modelo (Anthropic, Gemini, Ollama local…) es cambiar el starter y la configuración, no reescribir el chatbot.

### Cómo funciona

```
Usuario (chat.html)
   │  fetch POST /chat/messages  { "message": "reservame la sala A mañana de 10 a 11" }
   ▼
ChatController ──► ChatClient
                    ├─ system prompt (reglas de comportamiento)
                    ├─ historial de esta sesión (ChatMemory)
                    └─ lista de herramientas (@Tool de ChatToolsService)
                         │
                         ▼
                      Modelo OpenAI ── "necesito la fecha de hoy" ──► getCurrentDateTime()
                         │           ── "¿está libre?"             ──► listAvailableRooms(...)
                         │           ── "crear"                    ──► createBooking(...) ──► BookingService.create
                         ▼
                      Respuesta en lenguaje natural ──► usuario
```

**1) Las herramientas (`tool/ChatToolsService`)** son la única forma en que el modelo puede tocar el sistema. Cada una es un método Java con una descripción que el modelo lee:

```java
@Tool(description = "Lista las salas habilitadas para reservar, con su nombre y capacidad máxima. ...")
public List<RoomResponse> listRooms() {
   return roomService.findAllActive();
}
```

Hay herramientas de lectura (`getCurrentDateTime`, `listRooms`, `getBookingDurationRules`, `listMyBookings`, `listAvailableRooms`, `listAvailableSlotsForRoom`) y de escritura (`createBooking`, `cancelBooking`, `rescheduleBooking`, `unifyBookings`). **Todas delegan en los services**, así que las reglas de la sección 3 se aplican igual.

**2) El `ChatClient` (`config/ChatConfig`)** junta el system prompt, las herramientas y la memoria:

```java
return builder
      .defaultSystem(SYSTEM_PROMPT)
      .defaultTools(chatToolsService)
      .defaultAdvisors(MessageChatMemoryAdvisor.builder(chatMemory).build())
      .build();
```

**3) La memoria** usa como id de conversación el **id de la sesión HTTP**, así cada usuario tiene su propio historial:

```java
chatClient.prompt()
      .user(request.message())
      .advisors(a -> a.param(ChatMemory.CONVERSATION_ID, session.getId()))
      .toolContext(Map.of(ChatToolsService.BOOKING_CREATED_KEY, new AtomicBoolean(false)))
      .call()
      .content();
```

### Problemas concretos que aparecieron y cómo se resolvieron

| Problema | Solución |
|---|---|
| El modelo **puede inventar** salas, horarios o datos faltantes (ej. ponía "Reserva" como motivo) | El system prompt prohíbe inventar y obliga a preguntar; los datos reales siempre vienen de una herramienta |
| Las reglas de duración **las cambia un admin** y el modelo puede asumir valores viejos | Herramienta `getBookingDurationRules`, que lee la configuración vigente |
| Riesgo de que el modelo actúe **sobre reservas ajenas** | Las herramientas toman el usuario del `SecurityContext` (la sesión web), nunca de un parámetro que elija el modelo; el service valida el dueño |
| Pedidos del tipo "reservame todos los días hasta el domingo" creaban **muchas reservas de golpe** | `ToolContext` con un `AtomicBoolean` por mensaje: una segunda llamada a `createBooking` en el mismo mensaje se rechaza |


Es importante que las reglas de comportamiento en el prompt son **guías, no garantías**. Por eso lo crítico (propiedad, solapamientos, capacidad) **nunca se confía al prompt**: se valida en los services.

## 5. Capa de persistencia (Repository)

### 5.1 ¿Por qué PostgreSQL?

**El problema:** los datos son **claramente relacionales**: una reserva pertenece a una sala y a un usuario, y lo más importante del negocio (que dos reservas no se solapen) es una consulta sobre esas relaciones.

**En qué ayuda PostgreSQL en este proyecto:**
- **Integridad referencial:** las claves foráneas (`booking.room_id → rooms.id`, `booking.user_id → users.id`) impiden reservas "huérfanas" aunque haya un bug en el código.
- **UUID nativo** con `gen_random_uuid()`
- Es open source y está disponible como servicio administrado en Railway; en desarrollo se levanta con el archivo docker `docker-compose.yml`.

**Decisión de diseño:** la base guarda **estructura** (PK, FK, `NOT NULL`, `UNIQUE`), pero **no reglas de negocio**. No hay `CHECK` constraints de capacidad, duración ni grilla, todo eso lo controla el service

### 5.2 ¿Por qué Flyway? ¿Qué me permite?

El esquema de la base tiene que ser **el mismo** en mi máquina, en la de otro desarrollador y en producción (Railway). Crear las tablas a mano o dejar que Hibernate las genere (`ddl-auto=update`) hace que cada ambiente termine distinto, sin historial de qué cambió ni cuándo. **Flyway permite versionar el esquema junto con el código.** Los cambios son scripts SQL numerados en `src/main/resources/db/migration/`:

| Archivo | Contenido |
|---|---|
| `V1__create_initial_schema.sql` | Tablas `rooms`, `users`, `booking`, `booking_settings` |
| `V2__seed_initial_data.sql` | Datos iniciales: usuarios de prueba, salas A–E y la configuración de reservas (grilla de 30 min, máximo 6) |

- **Aplicación automática al arrancar.** Al levantar la app, Flyway ejecuta solo los scripts que todavía no se aplicaron en esa base y los registra en `flyway_schema_history`. Desplegar en Railway con una base vacía deja todo listo (tablas + datos semilla) sin pasos manuales.
- **Evolución segura.** Una migración ya aplicada no se modifica: cualquier cambio futuro es un `V3__...` nuevo, así cada base sabe exactamente en qué versión está.
- **Datos semilla reproducibles.** Cualquiera que clone el repo tiene salas, usuarios y reglas para probar desde el primer arranque.

**Flyway + Hibernate trabajando juntos:**
```properties
spring.jpa.hibernate.ddl-auto=validate
```
Flyway es el **único** que modifica el esquema; Hibernate solo **verifica** que las entidades Java coincidan con las tablas reales. Si alguien agrega un campo a una entidad y se olvida de la migración, la app **no arranca** y avisa.

### 5.3 Definición y mapeo del modelo: JPA + Hibernate + Spring Data

- **JPA / Hibernate** mapea cada clase a su tabla (`@Entity`, `@Table`, `@ManyToOne`…). El modelo tiene cuatro entidades: `Room`, `User`, `Booking` y `BookingSettings`.
- **Spring Data JPA** genera los repositorios a partir de interfaces. Hay tres niveles, del más simple al más específico:
  1. Métodos heredados (`findById`, `save`, `findAll(Pageable)`…).
  2. Consultas derivadas del nombre: `findByActiveTrueOrderByNameAsc()`, `existsByNameIgnoreCase(name)`.
  3. JPQL explícito, por ejemplo, para el **solapamiento**:

```java
@Query("""
      SELECT COUNT(b) > 0 FROM Booking b
      WHERE b.room.id = :roomId
        AND b.status = promtior.booking.backend.enums.BookingStatus.ACTIVE
        AND (:excludeBookingId IS NULL OR b.bookingId <> :excludeBookingId)
        AND b.startTime < :endTime
        AND b.endTime > :startTime
      """)
boolean existsOverlapping(...);
```
Utilizar esto permite abstraernos del motor de base de datos, porque es Hibernate el que traduce al dialecto SQL correcto. Esto permitiria que si quisiemos cambiar de motor de base de datos, podriamos hacerlo sin cambiar las consultas.

**Decisiones de modelado que resolvieron problemas puntuales:**
- **Validación duplicada en la entidad** (`@NotBlank`, `@Min`…, igual que en el DTO): Hibernate valida antes de cada `INSERT`/`UPDATE`. Es defensa en profundidad: si algún código arma una entidad sin pasar por un controller (por ejemplo, desde el chatbot), los datos inválidos igual no llegan a la base.
- **DTOs + MapStruct:** las entidades nunca salen por los endpoints. Se exponen records (`BookingResponse`, etc.) y MapStruct genera en compilación la conversión (por ejemplo, aplana `booking.room.name` a `roomName`). Así el contrato de la API no cambia si cambia el modelo interno, y se expone solo lo que se quiere exponer

### 5.4 ¿Por qué existe la clase `Auditable`?


Es una superclase común de la que heredan todas las entidades: es útil saber **cuándo se creó y cuándo se modificó por última vez** cada registro (por ejemplo, cuándo se reprogramó o canceló una reserva). Estas fecha las completa Spring Data automaticamente activado con `@EnableJpaAuditing` en `JpaAuditingConfig`

```java
@MappedSuperclass
@EntityListeners(AuditingEntityListener.class)
public abstract class Auditable {

   @CreatedDate
   @Column(name = "created_date", nullable = false, updatable = false)
   private LocalDateTime createdDate;

   @LastModifiedDate
   @Column(name = "last_modified_date", nullable = false)
   private LocalDateTime lastModifiedDate;
}
```